# Workspace Bootstrap (Colab)

Notebook template nay chi chua code mau. Du lieu thuc duoc lay tu backend qua launch token ngan han.

In [ ]:
import os
import requests

API_BASE = os.getenv("API_BASE", "http://localhost:8000/api/v1")
API_BASE = API_BASE.rstrip("/")
print("API_BASE =", API_BASE)

In [ ]:
launch_token = ""

try:
    from google.colab import output
    token_from_url = output.eval_js("new URL(window.location.href).searchParams.get('launch_token')")
    if token_from_url:
        launch_token = token_from_url
except Exception:
    pass

if not launch_token:
    launch_token = input("Paste launch_token from your app: ").strip()

if not launch_token:
    raise RuntimeError("Missing launch_token")

print("launch_token received")

In [ ]:
bootstrap = requests.post(
    f"{API_BASE}/colab/bootstrap",
    json={"token": launch_token},
    timeout=30,
)
bootstrap.raise_for_status()
cfg = bootstrap.json()
print("workspace_id:", cfg["workspace_id"])
print("user_id:", cfg["user_id"])
print("datasets:", len(cfg.get("datasets", [])))

In [ ]:
import pandas as pd

if not cfg.get("datasets"):
    print("No datasets attached to this workspace")
else:
    first_ds = cfg["datasets"][0]
    print("Loading:", first_ds["dataset_id"], first_ds["name"])
    df = pd.read_csv(first_ds["signed_url"])
    print(df.head())